# CNNs avec Keras sur des données de paysage

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'execution` et vérifiez que l'on est bien en Python 3 et que l'accélérateur matériel est configuré sur « GPU ».

In [ ]:
!nvidia-smi

## Téléchargement du dataset Landscape depuis un repo git

In [ ]:
!git clone https://github.com/shuuchuu/dataset-landscape.git
!ls dataset-landscape
print("***")
!ls -l dataset-landscape/seg_train
print("***")
!ls -l dataset-landscape/seg_pred

## Import de TensorFlow et des autres librairies nécessaires

In [ ]:
import itertools
import os
import pathlib
import random
import typing

import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy
import pandas
import PIL
import seaborn
import skimage.transform
import sklearn.utils
import sklearn.metrics
import tensorflow as tf
import keras as keras
import tqdm.notebook

## Préparation des données

Pour charger nos données, nous allons combiner plusieurs libraires : Pillow, NumPy & TensorFlow.

In [ ]:
INPUT_SHAPE = (150, 150, 3)


label_names = ["buildings", "forest", "glacier", "mountain", "sea", "street"]
label_to_index = {l: i for i, l in enumerate(label_names)}


def get_images(dir_path: pathlib.Path,
               size: int = INPUT_SHAPE[0],
               channels_first: bool = False,
               shuffle: bool = True,
               create_labels: bool = True,
               ) -> typing.Tuple[tf.Tensor, tf.Tensor]:
  images = []
  if create_labels:
    labels = []

  # On itère sur les sous-dossier de la racine : ils correspondent chacun à une
  # classe
  for subdir_path in tqdm.notebook.tqdm(
      list(dir_path.iterdir()), desc="Traitement des dossiers"):

    dir_name = subdir_path.name

    if create_labels:
      # On attribue le bon label en fonction du nom du dossier "labels"
      label = label_to_index.get(dir_name)

    # On ajoute chaque image du label (dossier) courant à notre dataset
    for image_path in tqdm.notebook.tqdm(
        list(subdir_path.iterdir()), desc=f"Dossier {dir_name}", leave=False):
      # Utilisation de PIL pour charger l'image
      images.append(
          numpy.array(PIL.Image.open(image_path).resize((size, size))))
      if create_labels:
        labels.append(label)

  images = tf.constant(numpy.array(images))
  if create_labels:
    labels = tf.constant(numpy.array(labels))

  if shuffle:
    perm = tf.random.shuffle(tf.range(images.shape[0]))
    images = tf.gather(images, perm)
    if create_labels:
      labels = tf.gather(labels, perm)

  if channels_first:
    images = tf.transpose(images, perm=(0, 3, 1, 2))

  if create_labels:
    return images, labels
  else:
    return images

## Appel à `get_images`

In [ ]:
images, labels = get_images(
    pathlib.Path("dataset-landscape") / "seg_train")

In [ ]:
print(f"Forme des images : {images.shape}")
print(f"Forme des labels : {labels.shape}")

seaborn.countplot(x=labels)
plt.title("Décomptes des différents labels")
plt.ylabel("Décompte")
plt.xlabel("Label")
plt.show()

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
f, ax = plt.subplots(5, 5, figsize=(15, 15))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    image = images[img_index]
    label = label_names[labels[img_index]]

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision par
    # ordinateur
    ax[i, j].imshow(image)
    ax[i, j].set_title(f"Exemple {img_index} ({label})")
    ax[i, j].axis('off')

## Création du modèle

Voici un exemple de CNN « minimaliste »

In [ ]:
# Initialisation et définition du modéle

# Le modèle est un empilement de couches où le flux de données est séquentiel
model = keras.models.Sequential()
# La première couche spécifie la taille de l'entrée
model.add(keras.layers.Input(shape=(150, 150, 3)))
# Une première couche de neurones de 1 convolutions de 3x3 pixels
model.add(keras.layers.Conv2D(1,
                              kernel_size=(3, 3),
                              activation="relu"))
# Une couche de max pooling
model.add(keras.layers.MaxPool2D(3,3))

# Une couche de manipulation des tenseurs : suppression de toutes les dimensions
# sauf celle de batch et une autre qui contient toutes les valeurs
model.add(keras.layers.Flatten())

# Une couche de sortie dense avec 6 neurones et un softmax comme activation
model.add(keras.layers.Dense(6, activation="softmax"))

# Compilation du modèle avec la définition de la fonction de perte
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Affichage d'un résumé du modèle
model.summary()

## Pouvez-vous expliquer les différents nombres de paramètres ?

### Solution


Premier layer de 1 convolutions : (taille du kernel) * (nb kernel) * (nb canaux en entrée) + (nb biais (= nb kernel)) = (3 * 3 ) * 1 * 3 + 1

Dernier layer dense : (input dim) * (output dim) + (nb biais) = 2401 * 6 + 6



## Apprentissage

Apprenons ce modèle sur nos données ! Dans un premier temps, nous entraînons sur une seule epoch pour simplement vérifier que notre modèle est opérationnel.

In [ ]:
# Apprentissage du modèle
training_history = model.fit(images, labels, epochs=1, validation_split=0.30)

## Améliorez cette performance

Inspirez-vous du modèle précédent en rajoutant des couches, en faisant des couches plus petites ou plus grosses.

Visez entre 10 et 20 itérations et mois de 1 minute par itération (pour des raisons évidentes).

On peut considérer l'utilisation d'une couche de dropout juste avant la dernière couche dense pour améliorer la régularisation.

On peut obtenir une précision supérieure à 70% sur la base de validation en un temps raisonnable.

La solution proposée prend $\approx$ 45 secondes par itération pendant 15 itérations et atteint aux alentour de 85% d'accuracy sur la base de validation.

In [ ]:
# Vos améliorations ici
model = keras.models.Sequential()
model.add(keras.layers.Input(shape=(150, 150, 3)))
model.add(keras.layers.Conv2D(10,
                              kernel_size=(3, 3),
                              activation="relu"))
model.add(keras.layers.MaxPool2D(3,3))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(6, activation="softmax"))
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Affichage d'un résumé du modèle
model.summary()

# Apprentissage du modèle
training = model.fit(images, labels, epochs=10, validation_split=0.30)


# Plot des métriques d'entraînement
def plot_metrics(history) -> None:
  plt.plot(training.history["accuracy"])
  plt.plot(training.history["val_accuracy"])
  plt.title("Accuracy du modèle")
  plt.ylabel("Accuracy")
  plt.xlabel("Epoch")
  plt.legend(["Entraînement", "Validation"], loc="upper left")
  plt.show()

  plt.plot(training.history["loss"])
  plt.plot(training.history["val_loss"])
  plt.title("Perte du modèle")
  plt.ylabel("Perte")
  plt.xlabel("Epoch")
  plt.legend(["Entraînement", "Validation"], loc="upper right")
  plt.show()


plot_metrics(training.history)

### Solution

In [ ]:
def conv() -> keras.layers.Conv2D:
  return keras.layers.Conv2D(filters=200,
                             kernel_size=3,
                             activation="relu",
                             kernel_initializer="orthogonal",
                             padding="same")


def pooling() -> keras.layers.MaxPooling2D:
  return keras.layers.MaxPooling2D(2, 2, padding="same")


def dropout(rate: float = 0.2) -> keras.layers.Dropout:
  return keras.layers.Dropout(rate)

def dense(units: int, activation: str = "relu") -> keras.layers.Dense:
  return keras.layers.Dense(units, activation=activation)


model = keras.models.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    pooling(),
    conv(),
    keras.layers.Flatten(),
    dropout(),
    dense(200),
    dropout(),
    dense(100),
    dropout(),
    dense(50),
    dropout(),
    dense(6, activation="softmax"),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Affichage d'un résumé du modèle
model.summary()

In [ ]:
# Apprentissage du modèle
training = model.fit(images,
                     labels,
                     epochs=15,
                     validation_split=0.30,
                     batch_size=128)

# Visualisation des métriques d'entrainement
plot_metrics(training.history)

Une implémentation de [LeNet](https://en.wikipedia.org/wiki/LeNet) :

In [ ]:
def conv(filters: int, padding: str) -> keras.layers.Conv2D:
  return keras.layers.Conv2D(filters=filters,
                             kernel_size=5,
                             padding=padding,
                             activation="sigmoid")


def pooling() -> keras.layers.MaxPooling2D:
  return keras.layers.MaxPooling2D()


def dense(units: int, activation: str = "sigmoid") -> keras.layers.Dense:
  return keras.layers.Dense(units, activation=activation)


le_net = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    conv(6, "same"),
    pooling(),
    conv(16, "valid"),
    pooling(),
    keras.layers.Flatten(),
    dense(120),
    dense(84),
    dense(6, activation="softmax")
], name="le_net")
le_net.summary()

le_net.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

training = le_net.fit(images,
                      labels,
                      epochs=15,
                      validation_split=0.30,
                      batch_size=128)
plot_metrics(training.history)

Une implémentation d'[AlexNet](https://en.wikipedia.org/wiki/AlexNet) :

In [ ]:
def conv(filters: int,
         kernel_size: int,
         padding: str = "same",
         strides: int = 1
         ) -> keras.layers.Conv2D:
  return keras.layers.Conv2D(filters=filters,
                             kernel_size=kernel_size,
                             strides=strides,
                             padding=padding,
                             activation="relu")


def pooling() -> keras.layers.MaxPooling2D:
  return keras.layers.MaxPooling2D(pool_size=3, strides=2)


def dropout(rate: float = 0.5) -> keras.layers.Dropout:
  return keras.layers.Dropout(rate)


def dense(units: int, activation: str = "relu") -> keras.layers.Dense:
  return keras.layers.Dense(units, activation=activation)


alex_net = keras.Sequential([
    keras.layers.Input(shape=(150, 150, 3)),
    conv(96, 11, "valid", 4),
    pooling(),
    conv(256, 5),
    pooling(),
    conv(384, 3),
    conv(384, 3),
    conv(384, 3),
    pooling(),
    keras.layers.Flatten(),
    dense(4096),
    dropout(),
    dense(4096),
    dropout(),
    dense(6, activation="softmax")
], name="alex_net")
alex_net.summary()

alex_net.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
                 loss="sparse_categorical_crossentropy",
                 metrics=["accuracy"])

training = alex_net.fit(images,
                        labels,
                        epochs=15,
                        validation_split=0.30,
                        batch_size=128)
plot_metrics(training.history)

Une implémentation de bloc [Inception](https://towardsdatascience.com/a-simple-guide-to-the-versions-of-the-inception-network-7fc52b863202) (pour constituer un réseau complet il faudrait en agencer plusieurs). Cette implémentation utilise [l'API fonctionnelle de Keras](https://keras.io/guides/functional_api/) :

In [ ]:
def block(inputs: keras.layers.Layer) -> keras.layers.Layer:
  # conv 1x1
  conv11 = keras.layers.Conv2D(64, 1, activation="relu")(inputs)

  # conv 1x1 puis conv 3x3
  conv11_3 = keras.layers.Conv2D(64, 1, activation="relu")(inputs)
  conv33_3 = keras.layers.Conv2D(64, 3, activation="relu", padding="same")(conv11_3)

  # conv 1x1 puis conv 5x5
  conv11_5 = keras.layers.Conv2D(64, 1, activation="relu")(inputs)
  conv55_5 = keras.layers.Conv2D(64, 5, activation="relu", padding="same")(conv11_5)

  # max pool 3x3 strides 1x1 puis conv 1x1
  max_pool = keras.layers.MaxPooling2D(3, 1, padding="same")(inputs)
  conv11_mp = keras.layers.Conv2D(64, 1, activation="relu")(max_pool)

  # concaténation
  concat = keras.layers.Concatenate()([conv11, conv33_3, conv55_5, conv11_mp])
  return concat

def inception() -> keras.models.Model:
  inputs = keras.layers.Input(shape=(150, 150, 3))
  x = block(inputs)
  x = keras.layers.Flatten()(x)
  outputs = keras.layers.Dense(6, activation="softmax")(x)


  model = keras.models.Model(inputs=[inputs], outputs=[outputs])
  model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                loss="sparse_categorical_crossentropy",
                metrics=["accuracy"])
  model.summary()
  return model


inception_model = inception()
training = inception_model.fit(images, labels, epochs=10, validation_split=0.30)

## Évaluation des performances sur l'ensemble de test

Dans le dossier `seg_test` se trouve un ensemble de données qui n'ont jamais été vues durant l'apprentissage.

On utilisera la méthode `evaluate(X, y)` du modèle pour évaluer la qualité de nos prédictions sur ce dataset.

In [ ]:
test_images,test_labels = get_images(
    pathlib.Path("dataset-landscape") / "seg_test")
model.evaluate(test_images, test_labels, verbose=1)

## Analyse d'erreur

On affiche la matrice de confusion puis on regarde des images mal classées.

In [ ]:
def analyze_preds(preds, labels):
  confusion_matrix = sklearn.metrics.confusion_matrix(labels, preds)
  seaborn.heatmap(confusion_matrix,
                  annot=True,
                  fmt="d",
                  cmap="rocket_r",
                  xticklabels=label_names,
                  yticklabels=label_names)
  plt.title("Matrice de confusion")
  plt.show()

  seaborn.countplot(x=list(map(lambda x: label_names[x], preds)))
  plt.title("Décomptes des classes prédites")
  plt.ylabel("Décompte")
  plt.xlabel("Class")
  plt.show()


test_pred = numpy.argmax(model.predict(test_images), axis=-1)
analyze_preds(test_pred, test_labels)

In [ ]:
def plot_mistakes(predicted_class: str, true_class: str) -> None:
  print(f"Prédiction : {predicted_class}, classe réelle : {true_class}")
  mistakes = test_images[(test_pred == label_names.index(predicted_class))
                         & (test_labels == label_names.index(true_class))]
  random_indexes = numpy.random.choice(mistakes.shape[0],
                                       size=min(mistakes.shape[0], 25),
                                       replace=False)
  grid_indexes = itertools.product(range(5), repeat=2)

  _, ax = plt.subplots(5, 5, figsize=(15, 15))
  for img_index, (i, j) in zip(random_indexes, grid_indexes):
    ax[i, j].imshow(mistakes[img_index])
    ax[i, j].axis("off")
  plt.show()

In [ ]:
# Plot les images prédites glacier alors qu'elles ont un label montagne
plot_mistakes("glacier", "mountain")

In [ ]:
# Plot les images prédites glacier alors qu'elles ont un label mer
plot_mistakes("glacier", "sea")

In [ ]:
# Plot les images prédites bâtiment alors qu'elles ont un label mer
plot_mistakes("buildings", "sea")

## Transfert d'apprentissage

In [ ]:
base_model = keras.applications.EfficientNetV2B0(include_top=False,
                                                 weights="imagenet",
                                                 input_shape=(150, 150, 3))

base_model.trainable = False

model = keras.Sequential(
    [base_model,
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(1024, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(256, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(64, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Conv2D(16, 1, activation="relu"),
     keras.layers.SpatialDropout2D(0.1),
     keras.layers.Flatten(),
     keras.layers.Dense(6, activation="softmax", kernel_regularizer="l2")])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

In [ ]:
training = model.fit(images,
                     labels,
                     epochs=15,
                     validation_split=0.30,
                     batch_size=512)

plot_metrics(training.history)

In [ ]:
model.evaluate(test_images, test_labels, verbose=1)
test_preds = numpy.argmax(model.predict(test_images), axis=-1)
analyze_preds(test_preds, test_labels)
plot_mistakes("glacier", "mountain")
plot_mistakes("glacier", "sea")
plot_mistakes("buildings", "sea")

## Prédire dans des condition « réelles »

Dans le dossier `seg_pred` se trouvent des images non-annotées. On ne peut donc pas évaluer correctement les performances sur cet ensemble.

Cependant, on peut afficher des photos et les probabilités que notre modèle attribue à chaque classe.

In [ ]:
pred_images = get_images(
    pathlib.Path("dataset-landscape") / "seg_pred",
    create_labels=False)
pred_images.shape

In [ ]:
# Création de la grille de sous-plots. On donne l'argument figsize pour agrandir
# la taille de la figure qui est petite par défaut
_, ax = plt.subplots(10, 5, figsize=(30, 45))

# On choisit 25 indices au hasard, sans replacement (on ne veut pas afficher la
# même image deux fois)
random_indexes = numpy.random.choice(pred_images.shape[0],
                                     size=(5, 5),
                                     replace=False)

for i in range(5):
  for j in range(5):
    img_index = random_indexes[i, j]
    # Récupération de l'image et prédiction de sa classe
    image = pred_images[img_index]
    probabilities = model.predict(image[None, ...])[0]
    predicted_class = label_names[numpy.argmax(probabilities)]

    # Affichage avec matplotlib et sa fonction imshow, très pratique en vision
    # par ordinateur
    ax[i * 2, j].imshow(image)
    ax[i * 2, j].set_title(f"Exemple {img_index}")
    ax[i * 2, j].axis('off')

    # Affichage de la distribution de prédiction sur la ligne d'en dessous
    ax[i * 2 + 1, j].bar(label_names, probabilities)

## Justesse en fonction de la probabilité maximale

Regardons maintenant si la justesse (*accuracy* en anglais) varie significativement en fonction de la probabilité maximale rendue par le modèle.

In [ ]:
test_probabilities = model.predict(test_images)

In [ ]:
def threshold_accuracy(probabilities: numpy.ndarray,
                       labels: tf.Tensor,
                       threshold: float
                       ) -> typing.Tuple[float, float, float, float]:
  labels = labels.numpy()
  predictions = probabilities.argmax(axis=-1)
  mask_above = probabilities.max(axis=-1) > threshold
  mask_below = ~mask_above

  n_above = mask_above.sum()
  n_below = mask_below.sum()

  if n_above:
    above = (predictions[mask_above] == labels[mask_above]).sum() / n_above
  else:
    above = 1.
  if n_below:
    below = (predictions[mask_below] == labels[mask_below]).sum() / n_below
  else:
    below = 1.
  return above, below, n_above / labels.shape[0], n_below / labels.shape[0]


accuracy_above, accuracy_below, ratio_above, ratio_below = threshold_accuracy(
    test_probabilities, test_labels, 0.99)
print("Justesse pour les prédictions du modèle qui ont une probabilité au "
      f"dessus du seuil ({ratio_above * 100:.2f}% des données) : "
      f"{accuracy_above:.2f}")
print("Justesse pour les prédictions du modèle qui ont une probabilité en "
      f"dessous du seuil ({ratio_below * 100:.2f}% des données) : "
      f"{accuracy_below:.2f}")

xs = numpy.arange(101)
accuracies = []
ratios_above = []
for x in xs:
  accuracy, _, ratio_above, _ = threshold_accuracy(test_probabilities,
                                                   test_labels,
                                                   x / 100)
  accuracies.append(accuracy)
  ratios_above.append(ratio_above)
plt.plot(xs, accuracies, label="Justesse")
plt.plot(xs, ratios_above, label="Rappel")
plt.xlabel("Seuil de prédiction")
plt.title("Justesse et rappel en fonction du seuil de prédiction")
plt.legend()
plt.show()

## Transfert d'apprentissage avec des transformeurs

In [ ]:
images, labels = get_images(pathlib.Path("dataset-landscape") / "seg_train",
                            size=224,
                            channels_first=True)

In [ ]:
import transformers


vit = transformers.TFAutoModel.from_pretrained(
    "google/vit-base-patch16-224-in21k", use_safetensors=False)

vit.trainable = False
inputs = keras.layers.Input((3, 224, 224))
x = keras.layers.Lambda(lambda t: t / 255)(inputs)
x = keras.layers.Normalization(mean=0.5, variance=0.5 ** 2)(x)

# Wrap the vit model call in a Lambda layer to handle the tensor type issue
x = keras.layers.Lambda(lambda x: vit(x).last_hidden_state)(x)

x = keras.layers.Dense(128, activation="relu")(x)
x = keras.layers.Dense(64, activation="relu")(x)
x = keras.layers.Dense(32, activation="relu")(x)
x = keras.layers.Flatten()(x)
outputs = keras.layers.Dense(6, activation="softmax")(x)
transferred_transformer = keras.Model(inputs=inputs,
                                      outputs=outputs,
                                      name="transferred_transformer")
transferred_transformer.compile(optimizer="adam",
                                metrics=["accuracy"],
                                loss="sparse_categorical_crossentropy")
transferred_transformer.summary()

In [ ]:
transferred_transformer.fit(images, labels, epochs=15, validation_split=0.3, batch_size=512)

In [ ]:
test_images, test_labels = get_images(
    pathlib.Path("dataset-landscape") / "seg_test",
    size=224,
    channels_first=True)